In [1]:
%pip install requests datasets

In [ ]:
%%writefile config.py

import torch

# Data
Data_path = "data/corpus.txt"
Train_split = 0.9

# Model
Block_size = 256
N_emb = 384
N_head = 6
N_layer = 6
Dropout = 0.2


assert N_emb % N_head == 0, (
    f"n_emb ({N_emb}) must be divisible by n_head ({N_head}) ")

# Training
Batch_size = 64
Max_iters = 5000
Eval_interval = 500
Eval_iters = 200
Learning_rate = 3e-4
Grad_clip = 1.0

#device
Device = "cuda" if torch.cuda.is_available() else "cpu"

Overwriting config.py


Overwriting config.py


In [62]:
%%writefile model.py

import torch
import torch.nn as nn
import torch.nn.functional as F
from config import Block_size, N_emb, N_head, N_layer, Dropout, Device


class Head(nn.Module):
    """Single causal: self:attention head"""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(N_emb, head_size, bias=False)
        self.query = nn.Linear(N_emb, head_size, bias=False)
        self.value = nn.Linear(N_emb, head_size, bias=False)
        
        self.register_buffer("tril", torch.tril(torch.ones(Block_size, Block_size)))
        self.dropout = nn.Dropout(Dropout)


    def forward(self, x):

        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        dim = k.shape[-1] 

        attn = q @ k.transpose(-2, -1) / (dim ** 0.5)
        attn = attn.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = attn @ v
        return out
    
class MultiHeadAttention(nn.Module):
    """Multiple heads in parallel, then project back to N_emb"""


    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(N_emb, N_emb) #projection after concatination
        self.dropout = nn.Dropout(Dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """"position-wise FFN: Linear-ReLu-Linear, with 4x inner dimension"""


    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_emb, 4 * N_emb),
            nn.ReLU(),
            nn.Linear(4 * N_emb, N_emb),
            nn.Dropout(Dropout),
        )

    def forward(self, x):
        return self.net(x)
    



    
class Block(nn.Module):
    """
    One transformer decoder block:
        x +=  MultiheadAttention(LayerNorm(x))  : communication step
        x += FeedForward(LayerNorm(x)) : computation step

    Nore: LayerNorm is applied BEFORE each sublayer (Pre-LN)
    The original paper used Post-LN but pre-LN trains more stably (why?)
    """

    def __init__(self):
        super().__init__()
        head_size = N_emb // N_head
        self.attn = MultiHeadAttention(N_head, head_size)
        self.ff = FeedForward()
        self.ln1 = nn.LayerNorm(N_emb)
        self.ln2 = nn.LayerNorm(N_emb)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x
    



class GPT(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, N_emb)
        self.pos_emb = nn.Embedding(Block_size,  N_emb)
        self.blocks = nn.Sequential(*[Block() for _ in range(N_layer)])
        self.ln_final = nn.LayerNorm(N_emb)
        self.head = nn.Linear(N_emb, vocab_size, bias=False)


    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_emb(idx)          # (B, T, N_emb)
        pos = self.pos_emb(torch.arange(T, device=Device)) # (T, N_emb)
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_final(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss            


    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Autoregressive generation: idx is (B,T) of current context"""

        for _ in range(max_new_tokens):
            idx_cond = idx[:, -Block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1) #(B, 1)
            idx = torch.cat([idx, next_tok], dim=1)

        return idx
    
    

Overwriting model.py


In [6]:
!mkdir -p data

In [52]:
%%writefile data/prepare.py

import os
import requests
import xml.etree.ElementTree as ET

def fetch_pubmed_abstracts(n_abstracts=100000, output_path="data/corpus.txt"):
    """
    Fetches Pubmed abstractions via NCBI E-utilites API ( no account needed).
    two steps:
        esearch ---> get a list of PMIDs matching a query
        efetch -----> retrieve the actual abstractions for thos PMIDs
    """

    Base   = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    # Broad query - covers most of biomediacl literature
    Query = "medical informatics[MeSH Major Topic] OR oncology[MeSH Major Topic]"
    Batch = 200 #efecth linit per request

    os.makedirs(os.path.dirname(output_path), exist_ok=True)


    print(f"Fetching {n_abstracts} PMIDs...") #?
    r = requests.get(Base + "esearch.fcgi" , params={
        "db" : "pubmed",
        "term" : Query,
        "retmax" : n_abstracts, 
        "retmode" : "json",
    })

    r.raise_for_status()
    pmids = r.json()["esearchresult"]["idlist"]
    print(f"Got {len(pmids)} PMIDs")


    all_text = []
    for i in range(0, len(pmids), Batch):
        batch = pmids[i : i + Batch]
        print(f"Fetching abstracts {i}-{i+Batch}...")
        r = requests.get(Base + "efetch.fcgi", params={
            "db" : "pubmed",
            "id" : ",".join(batch),
            "rettype" : "abstract",
            "retmode" : "xml",
        })

        r.raise_for_status()


        # Parse XML - abstracts live at .//AbstractText
        root = ET.fromstring(r.content)
        for node in root.findall(".//AbstractText"):
            text = (node.text or "").strip()
            if len(text) > 100: #??
                all_text.append(text)
        
    corpus = "\n\n".join(all_text)
    corpus = corpus.encode("ascii", errors="ignore").decode("ascii") 
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(corpus)

    print(f"\nDone. {len(all_text)} abstracts ----> {len(corpus):,} characters")
    print(f"Saved to {output_path}")

if __name__ == "__main__":
    fetch_pubmed_abstracts()



Overwriting data/prepare.py


In [63]:
%%writefile train.py

import torch
from config import *
from model import GPT

# ------------------------------------------------------------------
# Load corpus and build vocabulary
# ------------------------------------------------------------------
with open(Data_path, "r", encoding="utf-8") as f:
    text = f.read()

chars     = sorted(set(text))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}   # char → int
itos = {i: ch for i, ch in enumerate(chars)}   # int → char

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

# ------------------------------------------------------------------
# Train / val split
# ------------------------------------------------------------------
data  = torch.tensor(encode(text), dtype=torch.long)
n     = int(Train_split * len(data))
train_data = data[:n]
val_data   = data[n:]

# ------------------------------------------------------------------
# Batch sampling
# ------------------------------------------------------------------
def get_batch(split):
    data = train_data if split == "train" else val_data
    # Sample BATCH_SIZE random starting positions
    ix = torch.randint(len(data) - Block_size, (Batch_size,))
    x  = torch.stack([data[i     : i + Block_size    ] for i in ix])
    y  = torch.stack([data[i + 1 : i + Block_size + 1] for i in ix])
    return x.to(Device), y.to(Device)

# ------------------------------------------------------------------
# Loss estimation (averaged over EVAL_ITERS batches)
# ------------------------------------------------------------------
@torch.no_grad()
def estimate_loss(model):
    model.eval()
    losses = {}
    for split in ("train", "val"):
        L = torch.zeros(Eval_iters)
        for k in range(Eval_iters):
            x, y  = get_batch(split)
            _, loss = model(x, y)
            L[k]  = loss.item()
        losses[split] = L.mean().item()
    model.train()
    return losses

# ------------------------------------------------------------------
# Build model and optimizer
# ------------------------------------------------------------------
model = GPT(vocab_size).to(Device)
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print(f"Vocabulary: {vocab_size} characters")
print(f"Device: {Device}")

optimizer = torch.optim.AdamW(model.parameters(), lr=Learning_rate)

# ------------------------------------------------------------------
# Training loop
# ------------------------------------------------------------------
for step in range(Max_iters):

    if step % Eval_interval == 0:
        losses = estimate_loss(model)
        print(f"step {step:5d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f}")

    x, y = get_batch("train")
    logits, loss = model(x, y)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), Grad_clip)
    optimizer.step()

# Save checkpoint
torch.save({
    "model_state": model.state_dict(),
    "stoi"       : stoi,
    "itos"       : itos,
    "vocab_size" : vocab_size,
}, "checkpoint.pt")
print("Checkpoint saved.")

Overwriting train.py


In [64]:
!ls /content/
!ls /content/data/

checkpoint.pt  data	    model.py	 sample_data  trian.py
config.py      generate.py  __pycache__  train.py
corpus.txt  prepare.py


In [73]:
%%writefile generate.py

import torch
from config import Block_size, Device
from model import GPT

def load_model(checkpoint_path="checkpoint.pt"):
    ckpt = torch.load(checkpoint_path, map_location=Device)
    model = GPT(ckpt["vocab_size"]).to(Device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, ckpt["stoi"], ckpt["itos"]

def generate(prompt="", max_new_tokens=10000, temperature=0.8):
    model, stoi, itos = load_model()

    encode = lambda s: [stoi[c] for c in s if c in stoi]
    decode = lambda l: "".join([itos[i] for i in l])

    if prompt:
        context = torch.tensor(encode(prompt), dtype=torch.long, device=Device).unsqueeze(0)
    else:
        # Start from newline token — cold start
        context = torch.zeros((1, 1), dtype=torch.long, device=Device)

    output = model.generate(context, max_new_tokens=max_new_tokens, temperature=temperature)
    result = (decode(output[0].tolist()))
    print(result)
    print(f"Generated {len(result)} characters")

if __name__ == "__main__":
    generate(prompt="The patient presented with", max_new_tokens=500, temperature=0.8)

Overwriting generate.py


In [53]:
!python data/prepare.py

Fetching 100000 PMIDs...
Got 9999 PMIDs
Fetching abstracts 0-200...
Fetching abstracts 200-400...
Fetching abstracts 400-600...
Fetching abstracts 600-800...
Fetching abstracts 800-1000...
Fetching abstracts 1000-1200...
Fetching abstracts 1200-1400...
Fetching abstracts 1400-1600...
Fetching abstracts 1600-1800...
Fetching abstracts 1800-2000...
Fetching abstracts 2000-2200...
Fetching abstracts 2200-2400...
Fetching abstracts 2400-2600...
Fetching abstracts 2600-2800...
Fetching abstracts 2800-3000...
Fetching abstracts 3000-3200...
Fetching abstracts 3200-3400...
Fetching abstracts 3400-3600...
Fetching abstracts 3600-3800...
Fetching abstracts 3800-4000...
Fetching abstracts 4000-4200...
Fetching abstracts 4200-4400...
Fetching abstracts 4400-4600...
Fetching abstracts 4600-4800...
Fetching abstracts 4800-5000...
Fetching abstracts 5000-5200...
Fetching abstracts 5200-5400...
Fetching abstracts 5400-5600...
Fetching abstracts 5600-5800...
Fetching abstracts 5800-6000...
Fetching ab

In [66]:
!python train.py

Parameters: 10.81M
Vocabulary: 96 characters
Device: cuda
step     0 | train loss 4.7000 | val loss 4.7005
step   500 | train loss 1.9169 | val loss 1.9336
step  1000 | train loss 1.4288 | val loss 1.4452
step  1500 | train loss 1.2910 | val loss 1.3104
step  2000 | train loss 1.2023 | val loss 1.2271
step  2500 | train loss 1.1503 | val loss 1.1813
step  3000 | train loss 1.1123 | val loss 1.1392
step  3500 | train loss 1.0801 | val loss 1.1042
step  4000 | train loss 1.0527 | val loss 1.0817
step  4500 | train loss 1.0319 | val loss 1.0694
Checkpoint saved.


In [ ]:
!wc -c /content/data/corpus.txt

14793434 /content/data/corpus.txt


In [1]:
!python generate.py

Traceback (most recent call last):
  File "c:\Users\Saman\OneDrive\Desktop\Uni\Big Beautiful Biomedical Project\Fist GPT - Pubmed abstractions\generate.py", line 2, in <module>
    from config import Block_size, Device
  File "c:\Users\Saman\OneDrive\Desktop\Uni\Big Beautiful Biomedical Project\Fist GPT - Pubmed abstractions\config.py", line 27, in <module>
    Device = "cuda" if torch.cuda_is_available() else "cpu"
  File "c:\Users\Saman\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\__init__.py", line 2881, in __getattr__
    raise AttributeError(f"module '{__name__}' has no attribute '{name}'")
AttributeError: module 'torch' has no attribute 'cuda_is_available'
